# Final Project Study Plan

**Author:** Annie Lam

**Course:** HCDE 410

**Date:** November 2026

## Motivation & Problem Statement

Animal shelters are a community resource, and how well they're funded and staffed tends to
track the same neighborhood-level disparities seen in other public services. I want to ask:
**does an animal's outcome at an Austin city shelter, specifically time in shelter or whether it's euthanized, differ depending on the ZIP code it was found in, and does that pattern track neighborhood income?**

Practically, if certain ZIP codes consistently produce worse shelter outcomes, that's actionable
information for where to target spay/neuter outreach or low-cost vet clinics. Scientifically,
it's a human-centered case study in whether resource disparities well-documented elsewhere
(health, education) also show up somewhere easy to overlook: public animal welfare. It's the
same shape as the course's segregation/COVID-outcomes example: cross-referencing an outcomes
dataset against a geographic demographic dataset to check for a disparity. My preliminary exploration below confirms the data can support this kind of comparison, and an early look already complicates the simple story. See Overview & Research Questions below for the hypotheses this study plan now tests.

## Overview & Research Questions / Hypotheses

**Overview.** Building on the preliminary proposal, this study examines whether shelter outcomes
for dogs and cats at Austin Animal Center, specifically euthanasia and length of stay, vary
systematically with the median household income of the ZIP code tied to each animal's intake,
after accounting for animal-level factors that could independently explain outcome differences
(intake type, age, breed category).

**Primary research question:** For dogs and cats (including puppies and kittens), and after
controlling for intake type and age, is the median household income of an animal's ZIP code
associated with (a) the odds that it is euthanized, and (b) how long it stays in the shelter
before an outcome?

**Hypotheses:**
- **H1:** Controlling for intake type and age, dogs and cats from lower-income ZIP codes have
  higher odds of euthanasia than those from higher-income ZIP codes.
- **H2:** Controlling for the same factors, dogs and cats from lower-income ZIP codes have longer
  shelter stays than those from higher-income ZIP codes.
- **H0 (null, both):** there is no association between ZIP-code income and euthanasia odds or
  length of stay once these factors are controlled for.

My preliminary exploration initially complicated H1: the ZIP code with the highest raw
euthanasia rate (78701) is one of Austin's wealthier downtown ZIPs. Investigating it (Step 7
below) showed the pattern came from wildlife, mostly bats from the Congress Avenue Bridge
euthanized for rabies risk, not from income. That's why this study is scoped to dogs and cats,
and why H1 and H2 are phrased as controlled comparisons rather than raw ones (see Methodology).

**What I hope to determine:** whether a real, income-linked disparity in shelter outcomes exists
in Austin once animal-level factors are accounted for, and if so, how large it is, or whether no
pattern shows up once those factors are controlled, which would itself be a meaningful finding
given what the literature below suggests.

## Background & Related Work

Two directly relevant studies informed this design.

**Neal, S. M., & Kremer, T. (2024). Examining the relationship between social vulnerability and
animal shelter intakes and outcomes: Patterns and implications.** *Animals, 14*, Article 3166.
https://doi.org/10.3390/ani14223166. This study matched geocoded intake/adoption records from
seven U.S. shelters (85,515 animals) to the CDC's Social Vulnerability Index (SVI) by census
tract. It found that animals came disproportionately from high-vulnerability areas (the most
vulnerable quintile accounted for 47% of intakes despite being about 22.5% of tracts), but,
notably, **euthanasia rates were statistically similar across vulnerability quintiles**, so
animals from high-vulnerability areas were not euthanized at higher rates in that sample.

**Ly, L. H., Gordon, E., & Protopopova, A. (2021). Inequitable flow of animals in and out of
shelters: Comparison of community-level vulnerability for owner-surrendered and subsequently
adopted animals.** *Frontiers in Veterinary Science, 8*, 784389.
https://doi.org/10.3389/fvets.2021.784389. This study matched 21,270 BC SPCA surrender/adoption
records to the Canadian Index of Multiple Deprivation (CIMD) by community. It found that animals
(especially puppies and kittens) were disproportionately surrendered from more vulnerable
communities than they were adopted into, across three of four CIMD dimensions, using
Wilcoxon signed-rank tests with Bonferroni-corrected significance thresholds.

**How these shape my study:**
1. Both papers geocode animal records to area-level socioeconomic indices, the same strategy I
   use (ZIP-level median income rather than a composite deprivation index). This supports ZIP
   code as a workable geographic unit for this kind of question.
2. Both studies found socioeconomic measures matter more consistently for **where animals come
   from** (intake patterns) than for **euthanasia specifically**; Neal & Kremer found no
   meaningful euthanasia-rate gap by SVI quintile. My own data showed something similar: the raw
   ZIP comparison put its highest euthanasia rate in a wealthier downtown ZIP, and that turned
   out to be driven by wildlife intakes rather than income (Step 7). Simple area-level
   euthanasia differences can come from factors other than income itself, which is why H1 is
   framed as a controlled comparison rather than a raw one.
3. Both studies explicitly name the **ecological fallacy** (area-level deprivation is not the
   same as individual owner circumstances) as a limitation. That is the same concern my peer
   feedback raised about using ZIP-level income as a stand-in for pet-owner income. I adopt the
   same scoping language these papers use: this is a study of neighborhood-level association, not
   a claim about individual pet owners (see Methodology's Limitations below).
4. Methodologically, both papers used non-parametric group comparisons (Wilcoxon tests,
   quintile/correlation analysis) because they were comparing matched area pairs or ranked
   quintiles across many shelters. My question is structured differently: one shelter, one set of
   animals, a continuous income measure, and several animal-level covariates per record. Regression,
   which lets me hold multiple confounders constant at once, fits this data better than a
   quintile-comparison design, so my Methodology departs from directly replicating their approach.

## Data Used

**Primary: Austin Animal Center Intakes & Outcomes.** City of Austin's open data portal
publishes two linked datasets covering animals entering the shelter: an **Intakes** file (one
row per intake event, including a `Found Address` field with the stray's pickup location) and
an **Outcomes** file (one row per outcome, including `Outcome Status`, `Days in Shelter`, and
the matching `Intake Date`).

- Dataset pages: https://data.austintexas.gov/d/pyqf-r2dc (Intakes),
  https://data.austintexas.gov/d/gsvs-ypi7 (Outcomes)
- Direct CSV downloads: `.../api/views/{id}/rows.csv?accessType=DOWNLOAD`
- License: City of Austin Open Data Terms of Use (public reuse intended).
- **Data is not hosted in this repo.** I couldn't confirm the City of Austin's terms explicitly
  permit redistributing the raw files, so `data/` is empty in this repository; download both
  CSVs from the links above into a local `data/` folder to run this notebook.
- **Historical comparison file** (checked for feasibility, not used in the final analysis):
  Austin also publishes a 10/01/2013–05/05/2025 archive under separate dataset IDs (Intakes:
  data.austintexas.gov/d/wter-evkm, Outcomes: data.austintexas.gov/d/9t4d-g238). See Step 6
  below for why I didn't merge it in.

**Secondary: U.S. Census ACS**, 5-year median household income by ZIP Code Tabulation Area
(table B19013), via data.census.gov. A U.S. government work, public domain, no restrictions.
(Not yet joined in; see Preliminary Exploration below.)

**Why this fits:** answering the question requires one dataset of geographically-located animal
outcomes and one of neighborhood income to join against it.

**Ethical considerations:** the shelter data has no owner/personal information, keeping privacy
risk low. The bigger risk is interpretive. I need to avoid framing results as "this
neighborhood doesn't take care of its animals," which would stigmatize under-resourced
communities, especially since my own exploration below already shows the relationship isn't a
simple low-income-equals-worse-outcomes story. I'll frame findings around service access,
infrastructure, and animal-type mix, not residents' choices. I'll also watch for the ecological
fallacy: ZIP-level income isn't any individual owner's income, so conclusions stay at the
neighborhood level.

## Preliminary Data Exploration

I downloaded both CSVs directly from the City of Austin portal and ran a feasibility pass before
committing to this proposal. This is exactly what I did, including a join bug I found and fixed
along the way.

**Step 1: Load both files.**

In [ ]:
import pandas as pd
import re

intakes = pd.read_csv("data/Austin_Animal_Center_Intakes.csv")
outcomes = pd.read_csv("data/Austin_Animal_Center_Outcomes.csv")

print("Intakes:", intakes.shape)
print("Outcomes:", outcomes.shape)

**Step 2: First join attempt, on `Animal ID` alone.**

My first instinct was to join Outcomes to Intakes on `Animal ID` to pull in the `Found Address`
field. This was wrong: 688 animals have more than one intake record (repeat strays), so joining
on ID alone matches each outcome to *every* intake that animal ever had, silently inflating the
row count.

In [ ]:
# Naive join -- kept for transparency, this is what I caught and fixed.
naive_merge = outcomes.merge(
    intakes[["Animal ID", "Found Address"]],
    on="Animal ID",
    how="left"
)
print("Outcomes before join:", len(outcomes))
print("Rows after naive ID-only join:", len(naive_merge), "<- inflated, wrong")

dupe_ids = intakes["Animal ID"].value_counts()
print("Animal IDs with more than one intake record:", (dupe_ids > 1).sum())

**Step 3: Fixed join: match on `Animal ID` *and* the exact intake timestamp.**

Both files store a precise intake timestamp (`Source Date` in Intakes, `Intake Date` in
Outcomes), so joining on both columns together uniquely matches each outcome to the specific
intake visit it belongs to, instead of any visit by that animal.

In [ ]:
merged = outcomes.merge(
    intakes[["Animal ID", "Source Date", "Found Address"]],
    left_on=["Animal ID", "Intake Date"],
    right_on=["Animal ID", "Source Date"],
    how="left"
)

print("Outcomes before join:", len(outcomes))
print("Rows after corrected join:", len(merged), "<- matches, as it should")
print("Rows with a matched address:", merged["Found Address"].notna().sum(),
      f"({merged['Found Address'].notna().mean():.1%})")

**Step 4: Extract ZIP code from `Found Address`.**

`Found Address` is a free-text field (e.g. `"708 Simonetti Drive, AUSTIN, TX, 78748, United
States"`), not a clean ZIP column, so I pull the 5-digit ZIP out with a regex. Addresses with no
street-level detail (just `"TX, United States"`) correctly return no match and get dropped.

In [ ]:
def extract_zip(addr):
    if pd.isna(addr):
        return None
    match = re.search(r"\b(\d{5})\b", addr)
    return match.group(1) if match else None

merged["zip"] = merged["Found Address"].apply(extract_zip)

usable = merged["zip"].notna().sum()
print(f"Rows with a usable ZIP: {usable} / {len(merged)} ({usable/len(merged):.1%})")

**Step 5: Define euthanasia rate and look at the ZIP-level pattern.**

I flagged `Outcome Status` values of `"Euthanized"` or `"Euthanasia - RTO"` as the euthanasia
outcome, and grouped by ZIP (restricted to ZIPs with at least 50 outcomes, so small-sample noise
doesn't dominate).

In [ ]:
merged["euthanized"] = merged["Outcome Status"].isin(["Euthanized", "Euthanasia - RTO"])
print("Overall euthanasia rate:", round(merged["euthanized"].mean(), 4))

by_zip = (
    merged.dropna(subset=["zip"])
    .groupby("zip")
    .agg(n=("Animal ID", "count"),
         euth_rate=("euthanized", "mean"),
         median_days=("Days in Shelter", "median"))
    .query("n >= 50")
    .sort_values("euth_rate", ascending=False)
)

print(f"\n{len(by_zip)} ZIP codes with >= 50 outcomes")
print("\nHighest euthanasia-rate ZIPs:")
print(by_zip.head(5))
print("\nLowest euthanasia-rate ZIPs:")
print(by_zip.tail(5))

**What this shows:** the join and ZIP-extraction approach works cleanly (81% ZIP coverage
after the corrected join), and there's real variation in euthanasia rate across ZIP codes
(0.8%–27.3% among ZIPs with 50+ outcomes). The top outlier, 78701, is downtown Austin, one of
the city's *higher* income ZIPs, not a lower-income one. That's a useful early signal: the
relationship may not be a simple "lower income → worse outcomes" story, and could instead be
driven by animal-type mix (e.g., wildlife intakes downtown) rather than neighborhood resources.
I haven't joined Census income data yet (that's next), but this confirms the pipeline is
technically sound and already surfaces something worth investigating rather than assuming.

**Step 6: Checking the date range, and whether the older historical file is worth combining (per instructor feedback).**

Feedback pointed out that this dataset only covers a recent window, which is short for
comparing ZIP codes, and asked me to either flag that as a limitation or check whether Austin's
older 2013–2025 files have usable locations. I checked.

In [ ]:
intakes['Source Date'] = pd.to_datetime(intakes['Source Date'], format='mixed')
print('Date range:', intakes['Source Date'].min(), 'to', intakes['Source Date'].max())
print('Span (days):', (intakes['Source Date'].max() - intakes['Source Date'].min()).days)

**What this shows:** the current file covers only about 18 months (551 days), from May 2025
(when Austin switched to a new ShelterBuddy system) to November 2026. That's a real limitation
for comparing ZIP codes, since some ZIPs may simply have too few outcomes in that span to be
reliable.

Austin also publishes a separate historical dataset covering October 2013 to May 5, 2025 (before
the system switch), so I downloaded it and tested whether its location field is usable the same
way.

In [ ]:
hist_intakes = pd.read_csv("data/Historical_Intakes.csv")  # downloaded separately; see Data Used

def extract_zip(addr):
    if pd.isna(addr):
        return None
    match = re.search(r"\b(\d{5})\b", addr)
    return match.group(1) if match else None

hist_intakes["zip"] = hist_intakes["Found Location"].apply(extract_zip)
usable = hist_intakes["zip"].notna().sum()
print(f"Historical rows: {len(hist_intakes)}")
print(f"Rows with a usable ZIP: {usable} ({usable/len(hist_intakes):.1%})")

**Decision: not merging the historical file.** Only 11.9% of the historical records have an
extractable ZIP code, compared to 81% in the current file: the older `Found Location` field is
almost entirely street names without a ZIP (e.g., `"7405 Springtime in Austin (TX)"`). The
historical file also uses a different schema (no `Days in Shelter`, no `Euthanasia Reason`,
different intake-type categories), which would add real reconciliation work on top of a 7x lower
match rate. Given that, I'm not combining the two datasets. Instead, I'm carrying the short
18-month window forward as a stated limitation (see Methodology below) rather than diluting data
quality to extend it.

**Step 7: Investigating the 78701 outlier directly (per instructor feedback).**

Instructor feedback asked me to be able to explain what's driving the 78701 pattern, not just
flag it as an open question, so I looked at the actual records.

In [ ]:
z = merged[merged["zip"] == "78701"]
print("Found Address value counts in 78701 (top 5):")
print(z["Found Address"].value_counts().head(5))
print()
print("Animal Type breakdown in 78701:")
print(z["Type"].value_counts())

The top address is literally the Congress Avenue Bridge, Austin's famous bat colony, and
"Mammal" (which is how bats are classified in this dataset) makes up 78 of the 183 records in
this ZIP, more than Dogs or Cats individually.

In [ ]:
print("78701 euthanasia rate by Type:")
print(z.groupby("Type")["euthanized"].agg(["mean", "count"]))
print()
print("Euthanasia Reason counts among 78701 euthanized animals:")
print(z[z["euthanized"]]["Euthanasia Reason"].value_counts())

**What this shows:** 41 of the euthanized animals in 78701 were flagged "Rabies Risk",
consistent with bats found downtown being euthanized for public-health testing, which has
nothing to do with neighborhood income. To confirm this is really what's driving the outlier, I
recomputed the ZIP comparison restricted to just Dogs and Cats.

In [ ]:
PETS = ["Dog", "Cat", "Puppy", "Kitten"]

print("Citywide euthanasia rate, Mammal type only:",
      round(merged[merged["Type"]=="Mammal"]["euthanized"].mean(),4),
      "n =", (merged["Type"]=="Mammal").sum())
print("Citywide euthanasia rate, dogs and cats (incl. puppies/kittens):",
      round(merged[merged["Type"].isin(PETS)]["euthanized"].mean(),4),
      "n =", merged["Type"].isin(PETS).sum())
print("78701 euthanasia rate, dogs and cats only:",
      round(z[z["Type"].isin(PETS)]["euthanized"].mean(),4))
print()

dogcat = merged[merged["Type"].isin(PETS)].dropna(subset=["zip"])
by_zip_dc = (
    dogcat.groupby("zip")
    .agg(n=("Animal ID","count"), euth_rate=("euthanized","mean"))
    .query("n >= 50")
    .sort_values("euth_rate", ascending=False)
)
print(f"Dog/cat-only euthanasia rate by ZIP (n>=50), {len(by_zip_dc)} ZIPs, top 8:")
print(by_zip_dc.head(8))
print("\n78701 rank:", list(by_zip_dc.index).index("78701") + 1, "of", len(by_zip_dc))

**Conclusion:** citywide, "Mammal" records are euthanized at 47.0%, versus 3.9% for dogs and
cats (including puppies and kittens, which this dataset lists as separate types). Once I
restrict 78701 to dogs and cats, its euthanasia rate drops from the original 27.3% to 4.1%,
which ranks 14th of 36 ZIPs with 50+ outcomes, right around the citywide rate. This confirms the
outlier came from wildlife intakes near one bridge, not from an income-related pattern. Based on
this, I'm restricting the planned analysis in Methodology below to dogs and cats only.

## Methodology

**Scope.** Based on the investigation above, I'm restricting this analysis to **dogs and cats only** (`Type` = Dog, Cat, Puppy, or Kitten, since this dataset lists young animals separately). "Mammal" (largely bats near the Congress Avenue Bridge) and other wildlife categories are
euthanized for public-health reasons (rabies risk testing) that have nothing to do with
neighborhood income, and including them was entirely responsible for the 78701 outlier. Dropping
them removes a confound rather than controlling for it statistically, which is cleaner than
modeling it away.

**Address-type split (per instructor feedback).** `Found Address` means different things
depending on `Source Name` (the intake type): for owner surrenders, it's typically the owner's
own home address; for strays, ACO impounds, and community-animal intakes, it's wherever the
animal was physically found, which is not necessarily anyone's residence. Treating these the
same way would conflate "the neighborhood this owner lives in" with "the neighborhood this stray
happened to be found in." I'll analyze them as two separate subsets:
- **Owner-surrender subset** (`Source Name` in `Owner Surrender`, `ACO - Owner Surrender`,
  `Ambulance - Owner Surrender`): address plausibly reflects the surrendering owner's own
  neighborhood, so this is the subset most directly relevant to H1/H2 as phrased.
- **Stray/found subset** (`Stray`, `ACO - Impound`, `Community Animal - Program`, etc.): address
  reflects where the animal was found, which may still correlate with neighborhood
  under-resourcing but is a different claim than "this is where an owner who gave up their pet
  lives."

I'll report both subsets' results separately rather than pooling them under one coefficient.

**Variables.**
- *Outcome 1 (binary):* `euthanized`: 1 if `Outcome Status` is `"Euthanized"` or
  `"Euthanasia - RTO"`, 0 otherwise (already defined above; Dogs/Cats only per Scope above).
- *Outcome 2 (continuous):* `Days in Shelter`, right-skewed (see the ZIP-level medians above),
  so I'll check its distribution and likely analyze log(Days in Shelter + 1).
- *Primary predictor:* ZIP-level median household income, joined from ACS 5-year estimates
  (table B19013) by ZIP Code Tabulation Area (ZCTA). ZIP codes and ZCTAs aren't identical
  geographies, so I'll use the UDS Mapper ZIP-to-ZCTA crosswalk to match the `zip` field
  extracted above to the correct ZCTA before joining income, rather than assuming a ZIP code and
  its same-numbered ZCTA line up exactly.
- *Control variables:* Intake subtype (per the owner vs. stray split above), Age (computed from Date of Birth), and a grouped
  Breed category (raw breed has far too many distinct values to use directly). Animal Type is no
  longer a control variable, since Scope above restricts to Dog/Cat rather than modeling the
  wildlife confound statistically.

**Analytic approach.**
- *H1 (euthanasia):* logistic regression, run separately on the owner-surrender and
  stray/found subsets: `euthanized ~ income + intake_subtype + age`, reporting odds ratios and
  95% confidence intervals for the income coefficient in each subset.
- *H2 (length of stay):* linear regression on log-transformed `Days in Shelter`, same structure,
  also run separately by subset.
- Running both subsets separately lets me see whether an income association shows up differently
  depending on what the address actually represents, rather than assuming one coefficient means
  the same thing for both.
- I'll check for multicollinearity between income and the remaining controls, and report basic
  model diagnostics (residual plots for the linear model, a calibration/classification check for
  the logistic model) rather than relying on a single p-value as the whole answer.

**Presenting findings.**
- A descriptive bar chart of euthanasia rate and median days-in-shelter by income quartile (raw,
  for intuition), shown alongside the adjusted regression coefficients, for each subset. This
  mirrors how Neal & Kremer (2024) present both raw quintile comparisons and adjusted effect
  sizes.
- A coefficient plot (income's adjusted effect with its confidence interval) as the main result
  figure for the 5-minute presentation, with owner-surrender and stray/found subsets shown side
  by side.
- A scatter plot of ZIP income vs. euthanasia rate, restricted to Dog/Cat, to visually confirm
  the outlier is gone once Mammal/wildlife records are excluded.

**Limitations carried into this design:**
- *Ecological fallacy.* ZIP-level income describes the neighborhood, not the individual pet
  owner. All conclusions are scoped to neighborhood-level association, consistent with how the
  two papers above frame their own area-level findings. This is weaker for the stray/found
  subset, where the address may not correspond to anyone's residence at all, which is exactly why
  that subset is analyzed and reported separately.
- *Short date window.* The current (ShelterBuddy) dataset only covers about 18 months (May 2025
  to November 2026). I checked whether Austin's older 2013–2025 historical file could extend
  this window (see Step 6 above), but only 11.9% of its records have an extractable ZIP code,
  versus 81% in the current file, and its schema differs enough that combining them would
  introduce more noise than the extra years are worth. I'm treating the short window as an
  accepted limitation: some ZIP codes may have too few outcomes in 18 months to detect a real
  effect, and any pattern found should be read as describing this specific recent period, not a
  long-run trend.
- *Coverage/accuracy limits.* Only about 81% of outcome records have an extractable ZIP; records
  without a usable address are excluded, which could bias results if missingness isn't random,
  for example if certain intake types more often lack a detailed address.
- *Additional data considered.* I looked into whether a better proxy for "pet-owning household
  income" specifically exists (e.g., pet license/registration records). Austin's pet registration
  data isn't consistently public at the ZIP-level granularity this project needs, so I'm keeping
  ZIP-level ACS income as the practical choice and stating this limitation explicitly.

## Unknowns & Dependencies

1. **Join correctness.** I found and fixed a real join bug above (ID-only join silently
   duplicated rows for animals with repeat intakes, since Animal ID isn't unique per visit). I'm
   now joining on ID + exact intake timestamp, which matches the outcomes file's row count
   exactly, but I should still spot-check edge cases (e.g., same-animal intakes on the same
   timestamp, if any exist).
2. **ZIP coverage isn't complete.** About 81% of outcome records can be matched to a ZIP; the
   rest will be dropped or analyzed separately as "location unknown."
3. **Data license, resolved.** I could not find explicit language confirming the City of
   Austin's open data terms permit redistributing the raw files themselves, so I'm not including
   the raw CSVs in this public repo. The repo documents the direct download links instead (see
   Data Used), and anyone reproducing this notebook needs to download the files into a `data/`
   folder themselves.
4. **Short date window.** The current dataset only spans about 18 months (May 2025 to November
   2026). I checked whether Austin's older 2013–2025 file could extend this and found its
   location data too sparse to combine cleanly (11.9% ZIP match rate vs. 81%); see Step 6 and
   Methodology's Limitations.
5. **Address meaning differs by intake type.** `Found Address` means "owner's home" for
   surrenders but "where the animal was found" for strays; I'm now analyzing these as two
   separate subsets rather than pooling them (see Methodology).
6. **Census join not yet done.** I've confirmed the shelter-side pipeline works; joining in ACS
   income-by-ZIP data (via a ZIP-to-ZCTA crosswalk) is the next step, not yet attempted.

## AI use

- For python and markdown coding and debugging
- For grammar, summarizing, and proofreading this proposal and sources
- Help finding, citing, and summarizing related academic literature and other sources used
- Help investigating the 78701 outlier, researching this area (the Congress Avenue Bridge bat colony), and checking the records to identify the cause